# E3.7 · Building the capability

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E3.6 · Saying no, and saying yes with conditions](https://spbreed.github.io/cyber-commons/lessons/E3.6.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Write the interview loop for an agentic security engineer.

**Why a security engineer needs it.** Hiring for conceptual familiarity instead of practice. The control it builds is: interview questions that separate the two; internal transition paths.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You cannot hire this capability at the rate you need it, so most of it has to be built. Role definitions and honest ramp expectations are what stop that becoming an eighteen-month disappointment.

> **At CyberTravels.** CyberTravels cannot hire an identity engineer, a detection engineer and a harness engineer at the rate CyberTravels is changing. Most of that capability has to be built.

## 2 · The framework

```
   you cannot hire this at the rate you need it

   hire            2-3 people who have done it
   convert         AppSec, IAM, detection engineers
   ramp            6-9 months to independent, honestly

   role definitions first, or you interview for a job nobody can describe
```

Building the capability is a sequencing and hiring question, and the honest
version accounts for what you can **evidence** rather than what you can present.

The curve is deliberately unglamorous early. Quarter one produces an inventory
and identities — no dashboard, nothing to demo — and quarter three finally
produces numbers anyone outside the team finds interesting.

Programmes that invert it, doing evaluation and dashboards first, report high
numbers early and then spend a year discovering they cannot switch anything off
(E3.3). The inverted version is easier to fund and produces a capability that
fails its first real incident.

The four roles from E3.4 map onto the quarters, and the hire that unblocks the
most others is the identity engineer — which is not the one most teams hire
first.

## 3 · The procedure, as a skill

Two orders, the same work. The skill simulates both quarter by quarter: the correct one climbs to full coverage with nothing demoable until Q3, and the inverted one leads on capability for three quarters and finishes lower.

In [ ]:
# skills/programme/capability-build-order/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: capability-build-order
description: >-
  Compare a build order that produces control coverage against one that produces
  demos, quarter by quarter, and state what each has at the end of every period.
  Use when planning what to build first with a fixed team.
allowed-tools: Read, Grep, Glob
---

# Nothing demoable until Q3, and 100% coverage at Q4

Two orders, the same work. One builds the controls in dependency order and
produces coverage; the other builds the visible things first and produces
capabilities. The second looks better for three quarters and finishes lower, and
the only way to survive the first is to say in advance that it will look like
that.

## When to use this

Planning a multi-quarter build with a fixed team, and defending a plan whose
early quarters have nothing to show.

## Procedure

**1 — List the required controls and what each depends on.** Coverage is
measured against this list, so it has to exist before either order can be
scored.

**2 — Mark which controls are demoable.** Some produce something a stakeholder
can see; most do not. This is the honest input to the tension rather than a
complaint about it.

**3 — Simulate the dependency-respecting order.** Coverage per quarter and what
is demoable per quarter. Expect a flat, invisible start.

**4 — Simulate the demo-first order.** It will lead for several quarters on
capability and trail on coverage, and it will end lower because later controls
block on earlier ones.

**5 — Publish both, with the Q1 and Q2 warning explicit.** A plan that predicts
its own quiet period survives it; one that does not gets re-planned in month
four into the other order.

## Output contract

```json
{
  "required": ["str"],
  "demoable": {"str": false},
  "orders": [{"name": "str",
              "quarters": [{"q": "str", "built": ["str"], "coverage": 0.0, "demoable": 0}],
              "final_coverage": 0.0, "final_capabilities": 0}],
  "warning": "str"
}
```

## Failure modes

- **Building demoable things first.** It finishes lower.
- **Not warning about the quiet quarters.** The plan gets replaced in month
  four.
- **Coverage against a list that does not exist yet.** Write the required set
  first.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/programme/capability-build-order/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/programme/capability-build-order/scripts/capability_build_order.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Compare a build order that produces coverage against one that produces demos, quarter by quarter.

This is the executable half of the `capability-build-order` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
now = time.time(); DAY = 86400
REQUIRED = ["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2","DR-1","ST-1"]

QUARTERS = {
 "Q1 · inventory + identity": ["AC-1","AC-2"],
 "Q2 · containment":          ["AC-1","AC-2","SB-1","SB-2"],
 "Q3 · evidence + evaluation":["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2"],
 "Q4 · continuous + stop":    REQUIRED,
}
DEMOABLE = {"AC-1": False, "AC-2": False, "SB-1": False, "SB-2": False,
            "EV-1": False, "EV-2": True, "DR-1": True, "ST-1": True}

print(f"{'quarter':30s}{'coverage':>10}{'demoable':>11}")
print("-" * 54)
for q, done in QUARTERS.items():
    cov = len(done)/len(REQUIRED)
    demo = sum(1 for c in done if DEMOABLE[c])
    print(f"{q:30s}{cov:>10.0%}{demo:>11}")
print("\nQ1 and Q2 produce nothing demoable. That is the political problem, and")
print("it is why the inverted order keeps getting chosen.")

INVERTED = {
 "Q1 · evaluation + dashboard": ["EV-2","DR-1"],
 "Q2 · more evaluation":        ["EV-2","DR-1"],
 "Q3 · identity (finally)":     ["EV-2","DR-1","AC-1","AC-2"],
 "Q4 · containment":            ["EV-2","DR-1","AC-1","AC-2","SB-1","SB-2"],
}
CAPABILITY_AT = {
 "can revoke one agent":            {"AC-1"},
 "can attribute an action":         {"AC-1","EV-1"},
 "can bound a compromised agent":   {"SB-1","SB-2"},
 "can halt the fleet":              {"ST-1"},
 "can defend an accuracy number":   {"EV-2"},
}
def capabilities(done):
    return [c for c, need in CAPABILITY_AT.items() if need <= set(done)]

print(f"{'quarter':30s}{'coverage':>10}  capabilities")
print("-" * 90)
for q, done in INVERTED.items():
    print(f"{q:30s}{len(done)/len(REQUIRED):>10.0%}  {capabilities(done) or '—'}")

end_inverted = capabilities(INVERTED["Q4 · containment"])
end_correct  = capabilities(QUARTERS["Q4 · continuous + stop"])
print(f"\nafter four quarters:")
print(f"   inverted order: {len(end_inverted)} capabilities  {end_inverted}")
print(f"   correct order : {len(end_correct)} capabilities")
print("\nThe inverted programme spent a year and still cannot halt the fleet.")
assert len(end_correct) > len(end_inverted)

ROLES = {
 "harness engineer":   ("B2", {"EV-2"},                 "loop, verifier, eval"),
 "identity engineer":  ("A2", {"AC-1","AC-2","EV-1"},   "identity, delegation, act chains"),
 "detection engineer": ("D1", {"DR-1"},                 "agent telemetry and drift"),
 "GRC practitioner":   ("E1", {"SB-2","ST-1"},          "tiering, evidence, verification"),
}
def unblocks(role):
    delivered = ROLES[role][1]
    return [c for c, need in CAPABILITY_AT.items() if need & delivered]

print(f"{'role':22s}{'track':7s}{'controls':28s}unblocks")
print("-" * 92)
for role, (track, controls, what) in sorted(
        ROLES.items(), key=lambda kv: -len(unblocks(kv[0]))):
    print(f"{role:22s}{track:7s}{str(sorted(controls)):28s}{unblocks(role)}")

first = max(ROLES, key=lambda r: len(unblocks(r)))
print(f"\nhire first (unblocks the most): {first}")
print(f"hired first most often        : harness engineer")
assert first == "identity engineer"

print("\nplan that survives contact:")
for q, hire, deliver in [
 ("Q1", "identity engineer", "inventory + agent identities (AC-1, AC-2)"),
 ("Q2", "GRC practitioner",  "containment + tiering (SB-1, SB-2)"),
 ("Q3", "harness engineer",  "evidence + held-out evaluation (EV-1, EV-2)"),
 ("Q4", "detection engineer","drift + tested stop (DR-1, ST-1)"),
]:
    print(f"   {q}  hire {hire:20s}deliver {deliver}")

## What you just proved

The correct order climbs 25% → 50% → 75% → 100% coverage with nothing demoable until Q3. The inverted order reaches 75% after four quarters with 3 capabilities against the correct order's 5, and still cannot halt the fleet. The identity engineer unblocks the most capabilities and is named as the first hire.

## Your turn

Map your existing team onto the four roles. Most organisations have three of them under other names and are missing the identity one entirely — which is also the one that unblocks everything else.

---

**Next → [E3.8 · Resilience over perfection](https://spbreed.github.io/cyber-commons/lessons/E3.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*